# PEExcel with Python — Practical How-To

This notebook shows how to work with a **PEExcel project file** from Python using the `pexl` package.

It is intended for colleagues and Master's students who want to:

- read a PEExcel project file,
- inspect scenarios and variables,
- select inputs and results,
- compare variants,
- create pandas tables,
- modify inputs,
- export a project again,
- prepare data for plots and diagrams.

The examples use a PEExcel export workbook with `IN` and `OUT` sheets. Adjust the file path to your own project.

## 0. Setup

During development of `pexl`, `%autoreload` is useful because changes in the package are picked up without restarting the notebook.

For normal use, the two `%autoreload` lines can be omitted.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from pexl import Project

## 1. Open a PEExcel project file

A PEExcel workbook is loaded as one `Project`.

Each scenario or variant column in the workbook becomes one `Scenario` in Python.

In [ ]:
PROJECT_PATH = Path("../data/exports/ka_project_backup_v1_12.xlsx")

project = Project.from_excel(PROJECT_PATH)
project

A few useful project-level checks:

In [ ]:
print("Number of scenarios:", len(project))
print("Warnings:", len(project.warnings))
print("Project/group names:", project.project_names())

In [ ]:
# Show import warnings, if there are any
project.warnings[:10]

## 2. Work with scenarios

A Project usually contains a number of scenarios (Excel columns "Varianten"). They can be accessed as a list of Scenarios:

In [ ]:
project.scenarios

Scenarios can also be directly accessed by position in that list:

In [ ]:
s0 = project[1] # is equivalent to:
s1 = project.scenarios[1]

s0, s1

A scenario has three useful names:

- `column_name`: exact Excel scenario-column header; this is always unique and the safest identifier
- `project_name`: project or district grouping stored in PEExcel by the user (can be missing or wrong)
- `name`: short user provided scenario / variant name

The short scenario name is convenient for display, but it is not necessarily unique.

In [ ]:
print("Excel column name:", s0.column_name)
print("Project/group:", s0.project_name)
print("Scenario name:", s0.name)

You can also access a scenario directly by its exact Excel column name:

In [ ]:
column_name = project.column_names()[4]

scenario = project[column_name]
scenario

List available scenarios:

In [ ]:
for scenario in project:
    print(
        f"{scenario.column_name:45s}  "
        f"project={scenario.project_name!r}  "
        f"name={scenario.name!r}"
    )

## 3. Read individual PEExcel variables

If you already know the PEExcel variable name (column `var_name`), access it directly through `scenario.v`.

In [ ]:
print("Project description:", s0.v.project_description)
print("Total GFA:", s0.v.GFA_total)
print("Office GFA:", s0.v.GFA_office)

Metadata for the same variable is available through `scenario.meta`.

This is useful when you need labels, units, domains or other information for tables and plots.

In [ ]:
meta = s0.meta.GFA_total

print("Variable:", meta.var_name)
print("Label:", meta.label_de)
print("Unit:", meta.unit)
print("Source:", meta.source)
print("Domain:", meta.domain)
print("Measure:", meta.measure)

## 4. Select variables from a project

The most common workflow is to select groups of variables by their metadata.

Use:

- `project.select(...)` for all variables,
- `project.inn.select(...)` for input variables,
- `project.out.select(...)` for output variables.

In [ ]:
pe = project.out.select(
    domain="primary_energy_balance"
)

pe

You can combine several metadata filters:

In [ ]:
pe_demand = project.out.select(
    domain="primary_energy_balance",
    measure="demand",
)

pe_demand

Useful information about the result:

In [ ]:
print("Shape:", pe_demand.shape)
print("Scenarios:", len(pe_demand.scenarios))
print("Variables:", len(pe_demand.variables))
print()

print("Selected variable names:")
print(*pe_demand.variables.var_names, sep="\n")

The selected scenarios and variables behave like read-only collections:

In [ ]:
print("First scenario:")
print(pe_demand.scenarios[0])

print("\nFirst variable:")
print(pe_demand.variables[0])

print("\nFirst three variables:")
for meta in pe_demand.variables[:3]:
    print(meta.var_name, meta.unit)

In [ ]:
pe_demand.scenarios[1].view

## 5. Filter scenarios

Use `.where(...)` to restrict the scenarios in a project.

For example, select only scenarios belonging to one project or district:

In [ ]:
forsthaus = project.where(
    project_name="Forsthausgasse"
)

forsthaus

In [ ]:
for scenario in forsthaus.scenarios:
    print(scenario.column_name)

You can filter scenarios and variables together:

In [ ]:
comparison = (
    project.out
    .where(project_name="Forsthausgasse")
    .select(domain="primary_energy_balance")
)

comparison

The order is interchangeable. These two expressions describe the same data:

In [ ]:
a = (
    project.out
    .select(domain="primary_energy_balance")
    .where(project_name="Forsthausgasse")
)

b = (
    project.out
    .where(project_name="Forsthausgasse")
    .select(domain="primary_energy_balance")
)

print(a.shape)
print(b.shape)
print(a.to_records() == b.to_records())

## 6. Find available categories and values

Before writing filters, it is often useful to inspect what is available.

For output variables:

In [ ]:
print("OUT domains:", end="\n* ")
print(*project.out.domains(), sep="\n* ")

print("\nOUT measures:", end="\n* ")
print(*project.out.measures(), sep="\n* ")

For input variables:

In [ ]:
print("IN domains:", end="\n* ")
print(*project.inn.domains(), sep="\n* ")


In [ ]:
print("\nIN measures groups:", end="\n* ")
print(*project.inn.measures(), sep="\n* ")



In [ ]:
print("\nIN Entity groups:", end="\n* ")
print(*project.inn.entity_groups(), sep="\n* ")


To inspect the different values of one variable across scenarios, use `unique(...)` on the scenario selection.

In [ ]:
print(*project.view.scenarios.unique("project_name"))

This also works after filtering:

In [ ]:
subset = project.where(project_name="Campo Breitenlee")

print(
    subset.scenarios.unique(
        "preset_recorded_heating_system"
    )
)

## 7. Work with one scenario

For a single scenario use:

- `scenario.select(...)`,
- `scenario.inn.select(...)`,
- `scenario.out.select(...)`.

In [ ]:
scenario = project[1]

scenario.out

In [ ]:
heat = scenario.out.select(
    domain="heat_balance"
)

heat

The selected values can be returned as a dictionary:

In [ ]:
heat.to_var_dict()

Known selected variables can also be accessed directly from the view:

In [ ]:
qh_var_meta = scenario.meta.QH
print( qh_var_meta.icon, "QH:", round(heat.QH, 2), qh_var_meta.unit)

The same metadata exploration helpers are available:

In [ ]:
print("Scenario OUT domains:")
print(scenario.out.domains())

print("\nScenario IN entity groups:")
print(scenario.inn.entity_groups())

## 8. Convert a project selection to a pandas DataFrame

For most further analysis, a pandas DataFrame is the easiest format.

If `to_frame()` is available in your installed `pexl` version:

In [ ]:
df = comparison.to_frame()
df.head()

Alternatively, use the record representation:

In [ ]:
records = comparison.to_records()

df_records = (
    pd.DataFrame(records)
    .set_index("column_name")
)

df_records.head()

### Use short scenario names for presentation

The exact `column_name` should remain the technical identifier.

For tables or diagrams, you can replace the displayed index with the shorter scenario name:

In [ ]:
name_map = {
    scenario.column_name: scenario.name
    for scenario in comparison.scenarios
}

df_display = df_records.rename(index=name_map)
df_display.head()

Be careful: short scenario names can occur more than once. Use `column_name` whenever a unique identifier is required.

## 9. Compare scenarios

A common task is to compare selected output variables across several variants.

First restrict the scenarios and variables:

In [ ]:
comparison = (
    project.out
    .where(project_name="Forsthausgasse")
    .select(
        domain="primary_energy_balance"
    )
)

comparison

In [ ]:
df = comparison.to_frame()
df.head()

For a simple difference relative to the first scenario:

In [ ]:
reference = df.iloc[0]

difference = df.subtract(reference)
difference.head()

For relative differences in percent:

In [ ]:
relative_difference = (
    df.subtract(reference)
      .divide(reference)
      .multiply(100)
)

relative_difference.head()

## 10. Make a simple diagram

Once a view is converted to a DataFrame, standard pandas / matplotlib workflows can be used.

Example: compare selected primary-energy results by scenario.

In [ ]:
df_plot = comparison.to_frame()

# Replace the technical column names with short names for display
df_plot = df_plot.rename(
    index={
        s.column_name: s.name
        for s in comparison.scenarios
    }
)

ax = df_plot.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 5),
)

ax.set_xlabel("Scenario")
ax.set_ylabel("Value")
ax.legend(title="PEExcel variable")

plt.tight_layout()
plt.show()

For publication-quality or standard PEExcel diagrams, use the variable metadata (`label_de`, `unit`, `domain`, `measure`, ...) to define labels and groups rather than maintaining separate hard-coded variable lists.

## 11. Select inputs for one topic

Metadata filters are also useful for collecting related input parameters.

Example: residential-use inputs:

In [ ]:
residential_inputs = scenario.inn.select(
    entity_group="usage",
    entity_key="residential",
)

residential_inputs

In [ ]:
residential_inputs.to_var_dict()

Example: heating-related inputs:

In [ ]:
heating_inputs = scenario.inn.select(
    entity_group="heating"
)

heating_inputs

## 12. Change input values

Scenario values can be changed through `scenario.v`.

The following example is commented out so that running the notebook does not modify the project accidentally.

In [ ]:
scenario_edit = project[1]

print("Before:", scenario_edit.v.GFA_office)

# scenario_edit.v.GFA_office = 250.0

print("After:", scenario_edit.v.GFA_office)

After changing values, remember that PEExcel formulas are not recalculated by Python automatically unless your workflow explicitly runs Excel or another calculation step. Treat Python-side edits as changes to the project data.

## 13. Create a new project or scenario

A project can also be created directly in Python.

In [ ]:
from pexl.model.scenario import Scenario

new_project = Project()

new_scenario = Scenario(
    "Example Project | Variant A"
)

new_scenario.v.project_name = "Example Project"
new_scenario.v.project_scenario_name = "Variant A"
new_scenario.v.GFA_office = 250.0

new_project.add_scenario(new_scenario)

new_project

In [ ]:
new_project["Example Project | Variant A"]

When creating a new scenario, use a unique `column_name`. The conventional PEExcel form is:

```text
project_name | project_scenario_name
```

The exact column name remains the identifier used for import and export.

## 14. Export inputs or a complete IN/OUT project

`Project.to_excel(path)` retains the input-only default: IN and optional META.
Use `Project.to_excel(path, include_out=True)` for an IN/OUT workbook that can be loaded with `Project.from_excel()`.
Neither mode recalculates formulas or writes SIM2; hourly data attached to the original Project stays unchanged.
`include_derived=False` filters IN only, while OUT retains all stored result variables.

Use exports and schema bindings from a compatible model version: the workbook does not automatically switch the Python schema.


In [ ]:
OUTPUT_PATH = Path("../data/exports/new_project.xlsx")

# Uncomment to write and reload a complete IN/OUT export:
# new_project.to_excel(OUTPUT_PATH, include_out=True)
# restored_project = Project.from_excel(OUTPUT_PATH, unknown="raise")


For a normal workflow this gives:

```text
PEExcel project file
    ↓
Project.from_excel(...)
    ↓
select / compare / modify in Python
    ↓
Project.to_excel(...)
```

# Practical workflow examples

The following short examples combine the most common operations.

## A. Compare one result domain for one project

In [ ]:
view = (
    project.out
    .where(project_name="Forsthausgasse")
    .select(domain="primary_energy_balance")
)

df = view.to_frame()
df.head()

## B. Inspect all available variants of one project

In [ ]:
variants = project.where(
    project_name="Forsthausgasse"
)

for s in variants.scenarios:
    print(s.column_name, "->", s.name)

## C. Inspect available preset choices

In [ ]:
project.view.scenarios.unique(
    "preset_recorded_heating_system"
)

## D. Get selected values from one scenario

In [ ]:
scenario = project[1]

scenario.out.select(
    domain="primary_energy_balance"
).to_var_dict()

## E. Build a presentation table

In [ ]:
view = (
    project.out
    .where(project_name="Forsthausgasse")
    .select(domain="primary_energy_balance")
)

df = view.to_frame()

df = df.rename(
    index={
        s.column_name: s.name
        for s in view.scenarios
    }
)

df

# Planned standard-diagram API

The current workflow for diagrams is:

```python
view = project.out.where(...).select(...)
df = view.to_frame()
# plot with pandas / matplotlib / Plotly
```

For recurring PEExcel standard diagrams, a future convenience API should reduce this to a few user-facing commands while keeping the same project/scenario selection workflow.

A possible target usage could be:

```python
# FUTURE API — not implemented

view = project.where(
    project_name="Forsthausgasse"
)

view.diagram("primary_energy_balance")
```

or:

```python
# FUTURE API — not implemented

view.diagram(
    "heat_balance",
    scenarios=["Variant A", "Variant B"],
)
```

The intended user experience is that standard PEExcel diagrams already know:

- which PEExcel variables belong to the diagram,
- labels and units,
- stack/group assignment,
- display order,
- colors/patterns,
- any simple positive/negative split required for presentation.

Users should still be able to use `to_frame()` whenever they need a custom plot.

# Quick reference

| Task | Example |
|---|---|
| Open project | `project = Project.from_excel(path)` |
| First scenario | `project[0]` |
| Scenario by exact column | `project["Project | Variant"]` |
| Read one value | `scenario.v.GFA_total` |
| Read metadata | `scenario.meta.GFA_total` |
| Select variables | `project.select(domain="...")` |
| Select inputs | `project.inn.select(...)` |
| Select outputs | `project.out.select(...)` |
| Filter scenarios | `project.where(project_name="...")` |
| Combine both | `project.out.where(...).select(...)` |
| List domains | `project.out.domains()` |
| Unique scenario values | `project.view.scenarios.unique("project_name")` |
| DataFrame | `view.to_frame()` |
| Records | `view.to_records()` |
| One-scenario dictionary | `scenario.out.select(...).to_var_dict()` |
| Modify value | `scenario.v.GFA_office = 250.0` |
| Export | `project.to_excel(path)` |

## Suggested next exercises

1. Load a PEExcel project file of your own.
2. List all project/group names.
3. Select one project with `.where(...)`.
4. Find the available `OUT` domains.
5. Create a DataFrame for one domain.
6. Rename the scenario index for presentation.
7. Plot two or three selected variables.
8. Change one input value in a copied project and export it to a new file.

In [ ]:
import pexl

pexl.glossary


## 16. Excel IO tables versus analysis DataFrames

This example is self-contained and can be run independently of the previous cells.

- `Project.from_excel()` reads IN/OUT into `Project` and `Scenario`; SIM2, when present, is attached to its scenario.
- `project.inn.to_frame()` is an analysis table: scenarios are rows, variables are columns.
- `build_project_in_dataframe(project)` is an Excel serialization table: variables are rows, metadata and scenario identities are columns.
- `Project.to_excel()` handles the actual file write. DataFrames are temporary views, not a second project model.
- Schema CSV datasets use `io.csv_io`; they describe variables and are not scenario exports.

Internally, `io._project_tables` owns table mapping and `io.excel` owns workbook handling. The private helper module lets IN/OUT share mapping logic and lets tests inspect tables without opening Excel. It introduces no persistent format or public API; maintenance stays within the existing IO tests. Call the public Project methods for normal workflows.

The example below verifies both the unchanged input-only default and the IN/OUT export through the public Project API. Temporary files are removed afterward.

The optional `include_out` argument reuses the existing writer and table mapping instead of introducing a separate annual-export API or format. Its maintenance cost is limited to the OUT write path and regression tests for both modes.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from pexl import Project
from pexl.model.scenario import Scenario
from pexl.io.excel import build_project_in_dataframe

io_project = Project()
io_scenario = Scenario("IO example | Baseline")
io_scenario.v.project_name = "IO example"
io_scenario.v.project_scenario_name = "Baseline"
io_scenario.v.preset_recorded_heating_system = "Wärmepumpe"
io_scenario.v.QH = 42.5
io_project.add_scenario(io_scenario)

analysis_frame = io_project.inn.to_frame()
excel_frame = build_project_in_dataframe(io_project, base_columns=["var_name"])
assert analysis_frame.index.tolist() == [io_scenario.column_name]
assert excel_frame.columns.tolist() == ["var_name", io_scenario.column_name]

with TemporaryDirectory() as directory:
    output = io_project.to_excel(Path(directory) / "inputs.xlsx")
    with pd.ExcelFile(output) as book:
        assert book.sheet_names == ["IN", "META"]
        written = pd.read_excel(book, sheet_name="IN").set_index("var_name")
    assert written.loc["preset_recorded_heating_system", io_scenario.column_name] == "Wärmepumpe"

    annual_path = io_project.to_excel(Path(directory) / "annual.xlsx", include_out=True)
    restored = Project.from_excel(annual_path, unknown="raise")
    assert restored.column_names() == io_project.column_names()
    assert restored[0].v.QH == 42.5
    assert restored[0].v.preset_recorded_heating_system == "Wärmepumpe"
    assert not restored[0].timeseries.available

print("Analysis shape (scenarios x variables):", analysis_frame.shape)
print("Excel shape (variables x metadata/scenarios):", excel_frame.shape)
print("Input-only and IN/OUT exports verified; temporary files removed.")


## Excel-Arbeitsabläufe: Varianten, Migration und Batch

Die ausführbare Importanalyse oben verwendet die gespeicherten Excel-Ergebnisse. Die Berechnung selbst erfolgt in der Arbeitsmappe.

- Varianten anlegen, benennen und speichern; zum Bearbeiten laden und neu berechnen: `website/variants.qmd`.
- Migration: im alten Tool das Projekt exportieren, in einer separaten neuen Tooldatei importieren, Varianten/zusätzliche Profile prüfen und neu berechnen. Alte Datei und Export aufbewahren: `website/migration.qmd`.
- Viele Varianten im Blatt **Scen** per Batch berechnen und Rechenstatus prüfen; die konkrete Start-/Statusbedienung wird nach einer aktuellen Feldansicht ergänzt: `website/batch.qmd`.
- Danach den neuen Projektexport mit denselben `Project.from_excel`- und Analysefunktionen dieses Notebooks prüfen. Ein erfolgreicher Python-Import bestätigt keine vollständige Migration aller Excel-Zusatzdaten und keinen erfolgreichen Batch-Rechenlauf.

Für Migration und Scen sind noch aktuelle Dialogabbildungen erforderlich; diese Notebook-Ergänzung behauptet keine automatisierte Excel-Steuerung.

- Für Python-Simulationen der Gebäudetypologien aus der Basis `Default`: [Batch-How-to](howto_batch_simulate.ipynb). Es zeigt Inspektion, kleine Pilotauswahl, große Läufe, Checkpoints und portable Wiederholung über die öffentliche Batch-API.